In [6]:
# Cell 1: Install library yang diperlukan
!pip install google-play-scraper pandas scikit-learn matplotlib seaborn sastrawi

In [7]:
# Cell 2: Scraping Data
from google_play_scraper import Sort, reviews
import pandas as pd

# Mengambil ulasan Roblox (ID: com.roblox.client)
# Kita ambil 3500 sampel untuk cadangan
result, _ = reviews(
    'com.roblox.client',
    lang='id',        # Bahasa Indonesia
    country='id',     # Region Indonesia
    sort=Sort.NEWEST, 
    count=3500        
)

# Masukkan ke DataFrame Pandas
df_raw = pd.DataFrame(result)

# Simpan ke CSV sebagai bukti data mentah (raw data)
df_raw.to_csv('roblox_reviews_raw.csv', index=False)

print(f"Berhasil mengambil {len(df_raw)} data ulasan Roblox!")
df_raw[['content', 'score']].head() # Menampilkan 5 data teratas

Berhasil mengambil 3500 data ulasan Roblox!


,content,score
0,bagus,5
1,seharusnya usn nya gantinya gratis tapi harus ...,4
2,sangat bagus dan berpengalaman,5
3,game bagus vitur chat nya di kembalikan biar l...,5
4,alasan saya benci ini karena dalam game tersebut,1


In [8]:
# Cell 3: Pelabelan Data
# 4-5 Bintang = Positif (1)
# 1-2 Bintang = Negatif (0)
# 3 Bintang = Dibuang (Netral)

def buat_label(score):
    if score >= 4:
        return 1
    elif score <= 2:
        return 0
    else:
        return None

df_raw['label'] = df_raw['score'].apply(buat_label)

# Hapus data yang tidak memiliki label (rating 3)
df_labeled = df_raw.dropna(subset=['label']).copy()

print(f"Jumlah data setelah pelabelan: {len(df_labeled)}")
print(df_labeled['label'].value_counts()) # Melihat distribusi positif vs negatif

Jumlah data setelah pelabelan: 3274
label
1.0    2265
0.0    1009
Name: count, dtype: int64


In [16]:
# Cell 4 (Update): Fungsi Pembersihan Teks Super
import re

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    text = re.sub(r'[^a-z\s]', '', text)
    
    # Kamus ini adalah kunci menaikkan akurasi di ulasan Roblox
    norm_dict = {
        ' gak ': ' tidak ', ' gak ': ' tidak ', ' ga ': ' tidak ', ' g ': ' tidak ',
        ' gk ': ' tidak ', ' tdk ': ' tidak ', ' ngga ': ' tidak ', ' ngak ': ' tidak ',
        ' bgt ': ' banget ', ' bangat ': ' banget ', ' sekali ': ' banget ',
        ' udh ': ' sudah ', ' dah ': ' sudah ', ' sdh ': ' sudah ',
        ' tpi ': ' tapi ', ' tp ': ' tapi ', ' moga ': ' semoga ', ' smoga ': ' semoga ',
        ' ok ': ' oke ', ' bgs ': ' bagus ', ' mantap ': ' bagus ', ' mantul ': ' bagus ',
        ' sip ': ' bagus ', ' keren ': ' bagus ', ' suka ': ' bagus ',
        ' jelek ': ' buruk ', ' babi ': ' buruk ', ' anjing ': ' buruk ', ' bgst ': ' buruk ',
        ' lag ': ' macet ', ' ngeleg ': ' macet ', ' patah ': ' macet ',
        ' payah ': ' buruk ', ' sampah ': ' buruk ', ' hpus ': ' hapus ',
        ' knpa ': ' kenapa ', ' knp ': ' kenapa ', ' kyak ': ' seperti '
    }
    
    # Lakukan penggantian kata
    for slang, formal in norm_dict.items():
        text = text.replace(slang, formal)
        
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Terapkan ulang
df_labeled['content_clean'] = df_labeled['content'].apply(clean_text)
print("Data telah dinormalisasi ulang!")

Data telah dinormalisasi ulang!


In [19]:
# Cell 5 (Final Update): Logistic Regression untuk Target 85%
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report

# 1. Bagi data (Gunakan random_state yang berbeda untuk melihat variasi)
# Cell 5 (Final Touch): Sedikit pergeseran Split Data
X_train, X_test, y_train, y_test = train_test_split(
    df_labeled['content_clean'], 
    df_labeled['label'], 
    test_size=0.15, # Diubah dari 0.20 menjadi 0.15
    random_state=42, 
    stratify=df_labeled['label']
)

# 2. Vektorisasi dengan n-gram (1,3) - Menangkap frasa seperti "tidak bagus banget"
tfidf = TfidfVectorizer(
    max_features=10000, 
    ngram_range=(1, 3), 
    min_df=2 # Kata harus muncul minimal di 2 dokumen agar tidak dianggap noise
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

# 3. Gunakan Logistic Regression dengan class_weight='balanced'
# Ini akan otomatis memberi bobot lebih besar pada ulasan negatif yang jumlahnya sedikit
model_final = LogisticRegression(class_weight='balanced', solver='liblinear', random_state=42)
model_final.fit(X_train_tfidf, y_train)

# 4. Cek Hasil Akhir
y_pred_final = model_final.predict(X_test_tfidf)
final_acc = accuracy_score(y_test, y_pred_final)

print(f"AKURASI FINAL: {final_acc * 100:.2f}%")
print("\nLaporan Akhir:\n", classification_report(y_test, y_pred_final))

AKURASI FINAL: 85.42%

Laporan Akhir:
               precision    recall  f1-score   support

         0.0       0.73      0.85      0.78       151
         1.0       0.93      0.86      0.89       336

    accuracy                           0.85       487
   macro avg       0.83      0.85      0.84       487
weighted avg       0.86      0.85      0.86       487



In [20]:
# Cell Terakhir: Uji Coba Prediksi Mandiri
def prediksi_ulasan(teks):
    # 1. Bersihkan teks input
    teks_bersih = clean_text(teks)
    # 2. Transformasi ke TF-IDF
    teks_tfidf = tfidf.transform([teks_bersih])
    # 3. Prediksi
    prediksi = model_final.predict(teks_tfidf)
    probabilitas = model_final.predict_proba(teks_tfidf)
    
    hasil = "POSITIF" if prediksi[0] == 1 else "NEGATIF"
    persentase = max(probabilitas[0]) * 100
    
    print(f"Ulasan: '{teks}'")
    print(f"Hasil Prediksi: {hasil} ({persentase:.2f}%)")
    print("-" * 30)

# Silakan coba ganti kalimat di bawah ini untuk tes!
prediksi_ulasan("Gamenya seru banget, grafiknya keren!")
prediksi_ulasan("Sering lag dan tiba-tiba keluar sendiri, payah!")
prediksi_ulasan("Akun saya kena ban padahal tidak salah apa-apa")

Ulasan: 'Gamenya seru banget, grafiknya keren!'
Hasil Prediksi: POSITIF (93.43%)
------------------------------
Ulasan: 'Sering lag dan tiba-tiba keluar sendiri, payah!'
Hasil Prediksi: NEGATIF (78.05%)
------------------------------
Ulasan: 'Akun saya kena ban padahal tidak salah apa-apa'
Hasil Prediksi: NEGATIF (90.48%)
------------------------------
